# Full-weekend block redesign search
This notebook performs read-only analysis of `index.html` and searches for candidate even/odd 14-day arrays plus minimal overrides under the requested constraints.

In [ ]:
import re, json, random
from datetime import date, timedelta
from pathlib import Path

# 1) Parse existing Full weekend blocks (read-only)
text = Path('index.html').read_text(encoding='utf-8')
pat = re.compile(r"\{\s*label:\s*\"Full weekend blocks\"[\s\S]*?even:\s*\[(.*?)\],\s*odd:\s*\[(.*?)\],\s*overrides:\s*\{([\s\S]*?)\}\s*\}", re.M)
m = pat.search(text)
assert m, 'Full weekend blocks pattern not found'

def parse_arr(s):
    vals=[]
    for tok in s.split(','):
        t=tok.strip()
        if t=='null': vals.append(None)
        else: vals.append(int(t))
    return vals

cur_even = parse_arr(m.group(1))
cur_odd = parse_arr(m.group(2))
cur_overrides = dict(re.findall(r'\"(\d{4}-\d{2}-\d{2})\"\s*:\s*(null|0|1)', m.group(3)))
cur_overrides = {k:(None if v=='null' else int(v)) for k,v in cur_overrides.items()}

start = date(2026,4,9)
N = 364
dates = [start + timedelta(days=i) for i in range(N)]
weekday = [d.weekday() for d in dates]  # Mon=0..Sun=6


def expand(even, odd, overrides):
    workers=[]
    ov_idx = {dates.index(date.fromisoformat(k)):v for k,v in overrides.items()} if overrides else {}
    for i in range(N):
        arr = even if ((i//14)%2==0) else odd
        w = arr[i%14]
        if i in ov_idx:
            w = ov_idx[i]
        workers.append(w)
    return workers


def max_consec(workers, who):
    m=c=0
    for w in workers:
        if w==who:
            c+=1
            m=max(m,c)
        else:
            c=0
    return m


def exact4_off(workers, who):
    cnt=0
    s=0
    for w in workers+[who]:
        if w!=who:
            s+=1
        else:
            if s==4: cnt+=1
            s=0
    return cnt


def metrics(even,odd,overrides):
    w = expand(even,odd,overrides)
    a=sum(1 for x in w if x==0)
    r=sum(1 for x in w if x==1)
    o=sum(1 for x in w if x is None)
    aw=sum(1 for i,x in enumerate(w) if x==0 and weekday[i]>=5)
    rw=sum(1 for i,x in enumerate(w) if x==1 and weekday[i]>=5)
    paired=split=offwk=0
    for k in range(0,N,7):
        sat=w[k+2]; sun=w[k+3]
        if sat is None or sun is None: offwk+=1
        elif sat==sun: paired+=1
        else: split+=1
    return {
        'totals':(a,r,o),
        'max':(max_consec(w,0),max_consec(w,1)),
        'exact4':(exact4_off(w,0), exact4_off(w,1)),
        'weekend_counts':(aw,rw),
        'weekend_integrity':(paired,split,offwk)
    }

print('current metrics', metrics(cur_even, cur_odd, cur_overrides))

# 2) Search candidates
weekend_pairs=[(2,3),(9,10)]
weekend_idx={2,3,9,10}
nonweek=[i for i in range(14) if i not in weekend_idx]


def random_array():
    arr=[None]*14
    # strict full-weekend: Sat+Sun same person in each weekend pair, no null on weekends
    for a,b in weekend_pairs:
        v=random.randint(0,1)
        arr[a]=v; arr[b]=v
    null_pos=random.choice(nonweek)
    arr[null_pos]=None
    for i in nonweek:
        if i==null_pos: continue
        arr[i]=random.randint(0,1)
    return arr


def null_indices_year(even,odd):
    out=[]
    for i in range(N):
        arr=even if ((i//14)%2==0) else odd
        if arr[i%14] is None:
            out.append(i)
    return out


def score(m):
    diff=abs(m['exact4'][0]-m['exact4'][1])
    over=max(0,m['max'][0]-3)+max(0,m['max'][1]-3)
    split=m['weekend_integrity'][1]
    return (diff, over, split, m['exact4'][0]+m['exact4'][1])

best=[]
seen=set()
random.seed(42)

for _ in range(18000):
    even=random_array()
    odd=random_array()
    # base counts should be 169/169/26 before 2 overrides
    z=sum(x==0 for x in even)+sum(x==0 for x in odd)
    o=sum(x==1 for x in even)+sum(x==1 for x in odd)
    n=sum(x is None for x in even)+sum(x is None for x in odd)
    if not (z==13 and o==13 and n==2):
        continue

    null_idx=null_indices_year(even,odd)
    if len(null_idx)!=26:
        continue

    # choose one null->Aaron and one null->Randall
    local=None
    for _s in range(120):
        i,j=random.sample(null_idx,2)
        for a,b in ((0,1),(1,0)):
            overrides={dates[i].isoformat():a, dates[j].isoformat():b}
            m=metrics(even,odd,overrides)
            if m['totals']!=(170,170,24):
                continue
            sc=score(m)
            rec=(sc,even,odd,overrides,m)
            if (local is None) or sc<local[0]:
                local=rec
    if local is None:
        continue
    key=(tuple(local[1]),tuple(local[2]),tuple(sorted(local[3].items())))
    if key in seen:
        continue
    seen.add(key)
    best.append(local)

best.sort(key=lambda x:x[0])
print('candidate_count', len(best))
for i,(sc,even,odd,ov,m) in enumerate(best[:12],1):
    print(i, sc, m)
    print('even=',even)
    print('odd =',odd)
    print('ov  =',ov)

# feasibility signal under searched strict constraints
has_perfect = any(abs(x[4]['exact4'][0]-x[4]['exact4'][1])==0 for x in best)
print('has_perfect_equal_exact4_in_search=', has_perfect)
